# 07 — Entrenar M_seg (segmentación semántica foliar)

U-Net con encoder MobileNetV2 (ImageNet). Tres clases: `0=fondo`, `1=hoja_sana`, `2=hoja_enferma`.

Severidad continua = `px_enferma / px_hoja × 100%`.

Pseudo-labels generadas automáticamente por HSV (sin anotación manual).

Pipeline:
1. Generar máscaras HSV por imagen del dataset de patógenos.
2. Entrenar U-Net con Dice + CE loss.
3. Medir Dice y IoU por clase en cada época.
4. Diagnóstico bias/varianza + sweep de lambda.
5. Exportar `model_seg.tflite` (float32) y `model_seg_int8.tflite`.

Targets: `val_dice_enferma ≥ 0.60`, gap bias/varianza < 0.15.

In [ ]:
!pip install -q tensorflow albumentations scikit-learn matplotlib opencv-python-headless

In [ ]:
import tensorflow as tf
import numpy as np
import cv2
from pathlib import Path
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)
print('GPU:', tf.config.list_physical_devices('GPU'))

IMG_SIZE = (256, 256)
NUM_CLASSES = 3
BATCH = 8
EPOCHS_P1 = 20
EPOCHS_P2 = 40
LR_P1 = 1e-3
LR_P2 = 1e-4
L2_DEFAULT = 5e-5

DATA = Path('./splits/clasificacion_patogeno')
OUT = Path('./outputs')
OUT.mkdir(exist_ok=True)

In [ ]:
import numpy as np
import cv2

_HSV_GREEN_LOW  = np.array([20, 30, 30])
_HSV_GREEN_HIGH = np.array([90, 255, 255])
_HSV_LESION_RANGES = [
    (np.array([10, 50, 20]),  np.array([30, 255, 200])),
    (np.array([0,  50, 20]),  np.array([10, 255, 180])),
]
_HSV_NECROTIC_RANGES = [
    (np.array([0, 0, 0]),    np.array([180, 80, 60])),
]
_MORPH_KERNEL = np.ones((5, 5), np.uint8)


def gray_world_white_balance(img_rgb: np.ndarray) -> np.ndarray:
    result = img_rgb.astype(np.float32)
    avg = result.reshape(-1, 3).mean(axis=0)
    gray = avg.mean()
    scale = gray / (avg + 1e-6)
    result *= scale
    return np.clip(result, 0, 255).astype(np.uint8)


def clahe_luminance(img_rgb: np.ndarray, clip=2.0, grid=(8, 8)) -> np.ndarray:
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip, tileGridSize=grid).apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)


def chromatic_normalize(img_rgb: np.ndarray) -> np.ndarray:
    return clahe_luminance(gray_world_white_balance(img_rgb))


def make_pseudo_mask(img_rgb: np.ndarray) -> np.ndarray:
    bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)

    healthy = cv2.inRange(hsv, _HSV_GREEN_LOW, _HSV_GREEN_HIGH)

    diseased = np.zeros(hsv.shape[:2], dtype=np.uint8)
    for lo, hi in _HSV_LESION_RANGES + _HSV_NECROTIC_RANGES:
        diseased = cv2.bitwise_or(diseased, cv2.inRange(hsv, lo, hi))

    healthy  = cv2.morphologyEx(healthy,  cv2.MORPH_OPEN,  _MORPH_KERNEL)
    diseased = cv2.morphologyEx(diseased, cv2.MORPH_OPEN,  _MORPH_KERNEL)
    diseased = cv2.morphologyEx(diseased, cv2.MORPH_CLOSE, _MORPH_KERNEL)

    mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
    mask[healthy  > 0] = 1
    mask[diseased > 0] = 2
    return mask


def apply_leaf_mask(img_rgb: np.ndarray, mask: np.ndarray) -> np.ndarray:
    masked = img_rgb.copy()
    masked[mask == 0] = 0
    return masked


print('chromatic_normalize, make_pseudo_mask y apply_leaf_mask definidos OK')

In [ ]:
import time
import albumentations as A
from PIL import Image
from tensorflow.keras.utils import Sequence

DATA = globals().get('DATA', Path('./splits/clasificacion_patogeno'))
IMG_SIZE = globals().get('IMG_SIZE', (256, 256))
BATCH = globals().get('BATCH', 8)

_SEG_AUG = A.Compose([
    A.Rotate(limit=40, border_mode=0, p=0.6),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=15, p=0.4),
])


def _safe_read(fp, size):
    for _ in range(3):
        try:
            return np.array(Image.open(fp).convert('RGB').resize(size))
        except (OSError, IOError):
            time.sleep(0.3)
    return None


class SegSequence(Sequence):
    def __init__(self, directory, img_size, batch_size, augment, shuffle):
        self.img_size = img_size
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle
        self.samples = []
        for cls_dir in Path(directory).iterdir():
            if not cls_dir.is_dir():
                continue
            for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp'):
                self.samples.extend(str(p) for p in cls_dir.glob(ext))
        self.n = len(self.samples)
        if shuffle:
            np.random.shuffle(self.samples)

    def __len__(self):
        return max(1, self.n // self.batch_size)

    def __getitem__(self, idx):
        batch = self.samples[idx * self.batch_size:(idx + 1) * self.batch_size]
        imgs, masks = [], []
        for fp in batch:
            img = _safe_read(fp, self.img_size)
            if img is None:
                continue
            img = chromatic_normalize(img)
            mask = make_pseudo_mask(img)
            if self.augment:
                aug = _SEG_AUG(image=img, mask=mask)
                img, mask = aug['image'], aug['mask']
            imgs.append(img.astype(np.float32) / 255.0)
            masks.append(mask[..., np.newaxis].astype(np.float32))
        if not imgs:
            return np.zeros((1, *self.img_size, 3), np.float32), np.zeros((1, *self.img_size, 1), np.float32)
        return np.stack(imgs), np.stack(masks)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.samples)


train_gen = SegSequence(DATA / 'train', IMG_SIZE, BATCH, augment=True, shuffle=True)
val_gen   = SegSequence(DATA / 'val',   IMG_SIZE, BATCH, augment=False, shuffle=False)
print(f'Train: {train_gen.n} imgs | Val: {val_gen.n} imgs')

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

_inspect_paths = train_gen.samples[:3]
fig, axes = plt.subplots(len(_inspect_paths), 2, figsize=(8, 4 * len(_inspect_paths)))
for row, fp in enumerate(_inspect_paths):
    raw = np.array(Image.open(fp).convert('RGB').resize(IMG_SIZE))
    norm = chromatic_normalize(raw)
    axes[row, 0].imshow(raw)
    axes[row, 0].set_title('Original'); axes[row, 0].axis('off')
    axes[row, 1].imshow(norm)
    axes[row, 1].set_title('Gray-World + CLAHE'); axes[row, 1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
import tensorflow as tf

IMG_SIZE    = globals().get('IMG_SIZE', (256, 256))
NUM_CLASSES = globals().get('NUM_CLASSES', 3)
L2_DEFAULT  = globals().get('L2_DEFAULT', 5e-5)


def _decoder_block(x, skip, filters, l2):
    x = tf.keras.layers.UpSampling2D(2)(x)
    if skip is not None:
        x = tf.keras.layers.Concatenate()([x, skip])
    reg = tf.keras.regularizers.l2(l2)
    x = tf.keras.layers.Conv2D(filters, 3, padding='same', activation='relu', kernel_regularizer=reg)(x)
    x = tf.keras.layers.Conv2D(filters, 3, padding='same', activation='relu', kernel_regularizer=reg)(x)
    return x


def build_mseg(l2=L2_DEFAULT, dropout_rate=0.2):
    backbone = tf.keras.applications.MobileNetV2(
        input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet'
    )
    skip_names = [
        'block_1_expand_relu',
        'block_3_expand_relu',
        'block_6_expand_relu',
        'block_13_expand_relu',
    ]
    skips = [backbone.get_layer(n).output for n in skip_names]
    bottleneck = backbone.output

    x = _decoder_block(bottleneck, skips[3], 256, l2)
    x = _decoder_block(x,          skips[2], 128, l2)
    x = _decoder_block(x,          skips[1], 64,  l2)
    x = _decoder_block(x,          skips[0], 32,  l2)
    x = tf.keras.layers.UpSampling2D(2)(x)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    x = tf.keras.layers.Conv2D(NUM_CLASSES, 1, activation='softmax')(x)

    return tf.keras.Model(inputs=backbone.input, outputs=x, name='M_seg')


mseg = build_mseg()
mseg.summary(line_length=100)

In [ ]:
import tensorflow as tf

NUM_CLASSES = globals().get('NUM_CLASSES', 3)


def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_oh = tf.one_hot(tf.squeeze(tf.cast(y_true, tf.int32), -1), NUM_CLASSES)
    total = tf.constant(0.0)
    for c in range(NUM_CLASSES):
        p = y_pred[..., c]
        t = tf.cast(y_true_oh[..., c], tf.float32)
        inter = tf.reduce_sum(p * t)
        union = tf.reduce_sum(p) + tf.reduce_sum(t)
        total += 1.0 - (2 * inter + smooth) / (union + smooth)
    return total / tf.cast(NUM_CLASSES, tf.float32)


def seg_loss(y_true, y_pred):
    ce = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    return tf.reduce_mean(ce) + dice_loss(y_true, y_pred)


class DiceCoef(tf.keras.metrics.Metric):
    def __init__(self, class_id, name=None, **kwargs):
        super().__init__(name=name or f'dice_c{class_id}', **kwargs)
        self.class_id = class_id
        self.inter = self.add_weight('inter', initializer='zeros')
        self.union = self.add_weight('union', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        pred_c = tf.cast(tf.argmax(y_pred, -1) == self.class_id, tf.float32)
        true_c = tf.cast(tf.squeeze(y_true, -1) == self.class_id, tf.float32)
        self.inter.assign_add(tf.reduce_sum(pred_c * true_c))
        self.union.assign_add(tf.reduce_sum(pred_c) + tf.reduce_sum(true_c))

    def result(self):
        return (2 * self.inter + 1e-6) / (self.union + 1e-6)

    def reset_state(self):
        self.inter.assign(0.0)
        self.union.assign(0.0)


def compile_mseg(model, lr):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss=seg_loss,
        metrics=[
            DiceCoef(0, 'dice_bg'),
            DiceCoef(1, 'dice_sana'),
            DiceCoef(2, 'dice_enferma'),
        ],
    )


print('seg_loss, DiceCoef y compile_mseg definidos OK')

In [ ]:
from pathlib import Path
import tensorflow as tf

EPOCHS_P1 = globals().get('EPOCHS_P1', 20)
LR_P1     = globals().get('LR_P1', 1e-3)
OUT       = globals().get('OUT', Path('./outputs'))

for layer in mseg.layers:
    layer.trainable = False
for layer in mseg.layers:
    if layer.name.startswith('up_sampling') or layer.name.startswith('conv2d') or layer.name.startswith('concatenate') or layer.name.startswith('dropout'):
        layer.trainable = True

compile_mseg(mseg, LR_P1)

cbs_p1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_dice_enferma', mode='max', patience=8,
        restore_best_weights=True, verbose=1,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        str(OUT / 'model_seg_p1_best.keras'),
        monitor='val_dice_enferma', mode='max', save_best_only=True, verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7, verbose=1,
    ),
]

print('=== FASE 1: decoder (encoder congelado) ===')
h1 = mseg.fit(train_gen, validation_data=val_gen, epochs=EPOCHS_P1, callbacks=cbs_p1, verbose=1)

In [ ]:
from pathlib import Path
import tensorflow as tf

EPOCHS_P2 = globals().get('EPOCHS_P2', 40)
LR_P2     = globals().get('LR_P2', 1e-4)
BATCH     = globals().get('BATCH', 8)
OUT       = globals().get('OUT', Path('./outputs'))

for layer in mseg.layers:
    layer.trainable = True
for layer in mseg.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

steps = max(1, train_gen.n // BATCH) * EPOCHS_P2
lr_sched = tf.keras.optimizers.schedules.CosineDecay(LR_P2, steps, alpha=0.05)
compile_mseg(mseg, lr_sched)

initial_epoch = len(h1.history['loss'])

cbs_p2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_dice_enferma', mode='max', patience=10,
        restore_best_weights=True, verbose=1,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        str(OUT / 'model_seg_best.keras'),
        monitor='val_dice_enferma', mode='max', save_best_only=True, verbose=1,
    ),
]

print('=== FASE 2: fine-tune completo + Cosine LR ===')
h2 = mseg.fit(
    train_gen, validation_data=val_gen,
    epochs=initial_epoch + EPOCHS_P2, initial_epoch=initial_epoch,
    callbacks=cbs_p2, verbose=1,
)
mseg.save(OUT / 'model_seg.keras')
print('Modelo guardado en', OUT / 'model_seg.keras')

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

OUT = globals().get('OUT', Path('./outputs'))


def plot_training(h1, h2, title='M_seg'):
    loss_all  = h1.history['loss'] + h2.history['loss']
    vloss_all = h1.history['val_loss'] + h2.history['val_loss']
    dice_s    = h1.history.get('dice_sana', []) + h2.history.get('dice_sana', [])
    vdice_s   = h1.history.get('val_dice_sana', []) + h2.history.get('val_dice_sana', [])
    dice_e    = h1.history.get('dice_enferma', []) + h2.history.get('dice_enferma', [])
    vdice_e   = h1.history.get('val_dice_enferma', []) + h2.history.get('val_dice_enferma', [])

    ep  = range(1, len(loss_all) + 1)
    div = len(h1.history['loss'])
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(ep, loss_all, 'b-', label='train_loss')
    axes[0].plot(ep, vloss_all, 'r-', label='val_loss')
    axes[0].axvline(div, color='gray', linestyle='--', label='fine-tune')
    axes[0].set_title(f'{title} — Loss (Dice + CE)')
    axes[0].legend(); axes[0].grid(True)

    if dice_s:
        axes[1].plot(ep, dice_s,  'g-',  label='train_dice_sana')
        axes[1].plot(ep, vdice_s, 'g--', label='val_dice_sana')
    if dice_e:
        axes[1].plot(ep, dice_e,  'r-',  label='train_dice_enferma')
        axes[1].plot(ep, vdice_e, 'r--', label='val_dice_enferma')
    axes[1].axvline(div, color='gray', linestyle='--')
    axes[1].set_title(f'{title} — Dice por clase')
    axes[1].legend(); axes[1].grid(True)

    plt.tight_layout()
    plt.savefig(OUT / f'{title}_curves.png', dpi=120)
    plt.show()


def diagnose_bias_variance(h1, h2, model_name='M_seg'):
    train_loss = (h1.history['loss'] + h2.history['loss'])[-1]
    val_loss   = (h1.history['val_loss'] + h2.history['val_loss'])[-1]
    gap = val_loss - train_loss
    print(f'[{model_name}] train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | gap={gap:.4f}')
    if train_loss > 0.3:
        print('  ALTO BIAS — underfitting. Aumentar capacidad, mas epocas, menos regularizacion.')
    elif gap > 0.15:
        print('  ALTA VARIANZA — overfitting. Aumentar dropout, L2, data aug, early stopping.')
    else:
        print('  Bias/varianza balanceados.')
    dice_e_val = (h1.history.get('val_dice_enferma', [0]) + h2.history.get('val_dice_enferma', [0]))[-1]
    print(f'  val_dice_enferma final: {dice_e_val:.4f}  (objetivo >= 0.60) {"OK" if dice_e_val >= 0.60 else "REVISAR"}')


plot_training(h1, h2, title='M_seg')
diagnose_bias_variance(h1, h2, 'M_seg')

In [ ]:
from pathlib import Path
import tensorflow as tf

DATA      = globals().get('DATA', Path('./splits/clasificacion_patogeno'))
OUT       = globals().get('OUT', Path('./outputs'))
IMG_SIZE  = globals().get('IMG_SIZE', (256, 256))
BATCH     = globals().get('BATCH', 8)
EPOCHS_P1 = globals().get('EPOCHS_P1', 20)
LR_P1     = globals().get('LR_P1', 1e-3)

print('=== Sweep de lambda (L2) ===')
sweep_results = []
for l2_val in [0.0, 1e-5, 1e-4, 1e-3]:
    m = build_mseg(l2=l2_val)
    compile_mseg(m, LR_P1)
    h = m.fit(
        SegSequence(DATA / 'train', IMG_SIZE, BATCH, augment=True, shuffle=True),
        validation_data=SegSequence(DATA / 'val', IMG_SIZE, BATCH, augment=False, shuffle=False),
        epochs=15, verbose=0,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        ],
    )
    tl = h.history['loss'][-1]
    vl = h.history['val_loss'][-1]
    sweep_results.append((l2_val, tl, vl, vl - tl))
    print(f'  lambda={l2_val:.0e} -> train={tl:.4f} val={vl:.4f} gap={vl-tl:.4f}')

best = min(sweep_results, key=lambda r: r[2])
print(f'\nMejor lambda: {best[0]:.0e} (val_loss={best[2]:.4f})')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

_COLORS = np.array([[0,0,0], [34,197,94], [249,115,22]], dtype=np.uint8)


def visualize_predictions(generator, model, n=5):
    x_batch, y_batch = generator[0]
    preds = model.predict(x_batch[:n], verbose=0)
    pred_masks = np.argmax(preds, axis=-1)
    true_masks = y_batch[:n, ..., 0].astype(int)

    fig, axes = plt.subplots(n, 3, figsize=(12, n * 3))
    for i in range(min(n, len(x_batch))):
        img = (x_batch[i] * 255).astype(np.uint8)
        axes[i, 0].imshow(img)
        axes[i, 0].set_title('Original'); axes[i, 0].axis('off')
        axes[i, 1].imshow(_COLORS[true_masks[i]])
        axes[i, 1].set_title('Mascara HSV'); axes[i, 1].axis('off')
        axes[i, 2].imshow(_COLORS[pred_masks[i]])
        axes[i, 2].set_title('Pred M_seg'); axes[i, 2].axis('off')
    plt.tight_layout()
    plt.show()


visualize_predictions(val_gen, mseg, n=5)

In [ ]:
import tensorflow as tf
import numpy as np
from pathlib import Path

OUT = globals().get('OUT', Path('./outputs'))

converter = tf.lite.TFLiteConverter.from_keras_model(mseg)
tflite_float = converter.convert()
Path(OUT / 'model_seg.tflite').write_bytes(tflite_float)
size_f32 = Path(OUT / 'model_seg.tflite').stat().st_size / (1024 * 1024)
print(f'model_seg.tflite: {size_f32:.2f} MB (float32)')


def _rep_dataset():
    for i in range(min(50, len(val_gen))):
        x, _ = val_gen[i]
        for img in x:
            yield [img[np.newaxis].astype(np.float32)]


converter_int8 = tf.lite.TFLiteConverter.from_keras_model(mseg)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = _rep_dataset
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type  = tf.uint8
converter_int8.inference_output_type = tf.uint8
tflite_int8 = converter_int8.convert()
Path(OUT / 'model_seg_int8.tflite').write_bytes(tflite_int8)
size_int8 = Path(OUT / 'model_seg_int8.tflite').stat().st_size / (1024 * 1024)
print(f'model_seg_int8.tflite: {size_int8:.2f} MB (int8)')

print('\nCopiar a App/assets/models/seg/model_seg.tflite  (preferir int8 si < 4 MB)')